# Search Ranking & Query Understanding System

Architecture
------------
1. QueryUnderstanding
   - Normalizes and tokenizes the raw query
   - Classifies coarse query *intent/category* (e.g. "Electronics" vs "Fashion")
     using a simple keyword/TF-IDF classifier — in production this is typically a
     fine-tuned transformer or a large-scale multi-class classifier trained on
     click logs, but the interface/role in the pipeline is identical
   - Extracts structured signals (detected category, cleaned tokens) used
     downstream as ranking features

2. Feature Extraction (per query-document pair)
   - Text relevance: TF-IDF cosine similarity between query and document
   - Lexical match: exact keyword overlap ratio
   - Category match: does the query's inferred intent match the document's category
   - Popularity: click-through-rate / historical engagement prior
   - Freshness: recency of the document

3. LearningToRankModel
   - Pointwise Learning-to-Rank: trains a regressor (Gradient Boosted Trees) to
     predict a relevance label (0-3 graded relevance, like real search-quality
     rating scales) from the feature vector above
   - This is the same conceptual approach used by real LTR systems (e.g.
     LambdaMART-style models), simplified to a pointwise objective for clarity
     instead of pairwise/listwise loss

4. Evaluation
   - NDCG@K (Normalized Discounted Cumulative Gain) — the standard search-quality
     metric, since it accounts for both relevance grade AND rank position
     (a relevant doc ranked #1 is worth more than the same doc ranked #8)

Why this pipeline shape?
- Splitting "query understanding" from "ranking" mirrors real search architectures
  (e.g. Google/Amazon search stacks): query understanding narrows and enriches the
  candidate set, and a separate ranker scores/orders whatever candidates make it
  through retrieval.

In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import LogisticRegression

In [2]:
CATEGORIES = ["Electronics", "Books", "Fashion", "Home", "Sports"]

In [3]:
# ----------------------------------------------------------------------------
# 1. Synthetic catalog + query log (stand-in for a real product catalog + search logs)
# ----------------------------------------------------------------------------
def generate_catalog(n_docs=300, seed=42):
    rng = np.random.default_rng(seed)
    category = rng.choice(CATEGORIES, size=n_docs)
    popularity = rng.beta(2, 5, size=n_docs)          # historical CTR-like prior, skewed low
    recency_days = rng.exponential(60, size=n_docs).clip(0, 720)

    # Category-specific vocabulary so TF-IDF/intent classification has real signal
    # to work with (mirrors how real product titles contain descriptive keywords,
    # not just the category name).
    keyword_pool = {
        "Electronics": ["wireless", "headphones", "bluetooth", "speaker", "charger",
                         "laptop", "smart", "watch", "earbuds", "usb"],
        "Books": ["novel", "mystery", "cookbook", "recipes", "self", "help",
                  "fiction", "sci-fi", "bestseller", "paperback"],
        "Fashion": ["running", "shoes", "jacket", "winter", "leather", "wallet",
                    "dress", "summer", "sneakers", "denim"],
        "Home": ["kitchen", "knife", "led", "lamp", "storage", "boxes",
                 "pillow", "memory", "foam", "furniture"],
        "Sports": ["yoga", "mat", "tennis", "racket", "dumbbell", "cycling",
                   "helmet", "fitness", "gym", "training"],
    }
    titles = []
    for cat in category:
        words = rng.choice(keyword_pool[cat], size=4, replace=False)
        titles.append(f"{' '.join(words)} - durable high quality {cat.lower()} item")

    return pd.DataFrame({
        "doc_id": np.arange(n_docs),
        "category": category,
        "title": titles,
        "popularity": popularity,
        "recency_days": recency_days,
    })


def generate_query_log(catalog, n_queries=150, seed=42):
    rng = np.random.default_rng(seed)
    query_templates = {
        "Electronics": ["wireless headphones", "laptop charger", "bluetooth speaker", "smart watch"],
        "Books": ["mystery novel", "cook book", "self help book", "sci fi book"],
        "Fashion": ["running shoes", "winter jacket", "leather wallet", "summer dress"],
        "Home": ["kitchen knife set", "led desk lamp", "storage boxes", "memory foam pillow"],
        "Sports": ["yoga mat", "tennis racket", "dumbbell set", "cycling helmet"],
    }
    rows = []
    for qid in range(n_queries):
        cat = rng.choice(CATEGORIES)
        query = rng.choice(query_templates[cat])
        rows.append((qid, query, cat))
    return pd.DataFrame(rows, columns=["query_id", "query_text", "true_intent_category"])

In [4]:
# ----------------------------------------------------------------------------
# 2. Query Understanding module
# ----------------------------------------------------------------------------
class QueryUnderstanding:
    def __init__(self):
        self.vectorizer = TfidfVectorizer(stop_words="english")
        self.intent_clf = LogisticRegression(max_iter=1000)

    def fit(self, catalog):
        """Train intent classifier on catalog titles as a proxy for labeled query intent data."""
        X = self.vectorizer.fit_transform(catalog.title)
        y = catalog.category
        self.intent_clf.fit(X, y)
        return self

    def understand(self, query_text):
        clean = query_text.lower().strip()
        tokens = clean.split()
        vec = self.vectorizer.transform([clean])
        intent = self.intent_clf.predict(vec)[0]
        intent_confidence = self.intent_clf.predict_proba(vec).max()
        return {"clean_query": clean, "tokens": tokens, "intent_category": intent,
                "intent_confidence": intent_confidence}

In [5]:
# ----------------------------------------------------------------------------
# 3. Feature extraction for query-document pairs
# ----------------------------------------------------------------------------
class FeatureExtractor:
    def __init__(self, catalog):
        self.catalog = catalog.reset_index(drop=True)
        self.vectorizer = TfidfVectorizer(stop_words="english")
        self.doc_vectors = self.vectorizer.fit_transform(catalog.title)

    def extract(self, query_understanding, candidate_docs):
        query_vec = self.vectorizer.transform([query_understanding["clean_query"]])
        idx = candidate_docs.index.values
        text_sim = cosine_similarity(query_vec, self.doc_vectors[idx]).flatten()

        query_tokens = set(query_understanding["tokens"])
        lexical_overlap = candidate_docs.title.apply(
            lambda t: len(query_tokens & set(t.lower().split())) / max(len(query_tokens), 1)
        ).values

        category_match = (candidate_docs.category == query_understanding["intent_category"]).astype(int).values
        popularity = candidate_docs.popularity.values
        freshness = 1 / (1 + candidate_docs.recency_days.values / 30)  # decays with age

        features = pd.DataFrame({
            "text_similarity": text_sim,
            "lexical_overlap": lexical_overlap,
            "category_match": category_match,
            "popularity": popularity,
            "freshness": freshness,
        }, index=candidate_docs.index)
        return features

In [6]:
# ----------------------------------------------------------------------------
# 4. Synthetic graded relevance labels (0=irrelevant .. 3=highly relevant)
#    In production these come from human relevance judgments or click models.
# ----------------------------------------------------------------------------
def label_relevance(features, category_match_weight=1.5):
    score = (
        2.0 * features.text_similarity
        + category_match_weight * features.category_match
        + 0.5 * features.popularity
        + 0.3 * features.lexical_overlap
    )
    # Bucket into graded relevance 0-3
    bins = pd.qcut(score, q=4, labels=[0, 1, 2, 3], duplicates="drop")
    return bins.astype(int)

In [7]:
# ----------------------------------------------------------------------------
# 5. Learning-to-Rank model
# ----------------------------------------------------------------------------
class LearningToRankModel:
    def __init__(self, seed=42):
        self.model = GradientBoostingRegressor(
            n_estimators=200, max_depth=3, learning_rate=0.05, random_state=seed
        )

    def fit(self, X, y):
        self.model.fit(X, y)
        return self

    def rank(self, features, candidate_docs, top_k=10):
        scores = self.model.predict(features)
        ranked = candidate_docs.copy()
        ranked["ltr_score"] = scores
        return ranked.sort_values("ltr_score", ascending=False).head(top_k)

In [8]:
# ----------------------------------------------------------------------------
# 6. NDCG@K evaluation
# ----------------------------------------------------------------------------
def ndcg_at_k(relevance_scores, k=10):
    relevance_scores = np.asarray(relevance_scores)[:k]
    discounts = 1 / np.log2(np.arange(2, len(relevance_scores) + 2))
    dcg = np.sum(relevance_scores * discounts)
    ideal = np.sort(relevance_scores)[::-1]
    idcg = np.sum(ideal * discounts)
    return dcg / idcg if idcg > 0 else 0.0


if __name__ == "__main__":
    catalog = generate_catalog(n_docs=300)
    query_log = generate_query_log(catalog, n_queries=150)

    qu = QueryUnderstanding().fit(catalog)
    extractor = FeatureExtractor(catalog)

    # Build a training set: for each query, extract features vs full catalog + relevance labels
    all_X, all_y = [], []
    for _, q_row in query_log.iterrows():
        understanding = qu.understand(q_row.query_text)
        feats = extractor.extract(understanding, catalog)
        labels = label_relevance(feats)
        all_X.append(feats)
        all_y.append(labels)

    X_train = pd.concat(all_X, ignore_index=True)
    y_train = pd.concat(all_y, ignore_index=True)

    ltr = LearningToRankModel().fit(X_train, y_train)

    # --- Demo: rank results for a new query ---
    test_query = "wireless headphones"
    understanding = qu.understand(test_query)
    print(f"=== Query Understanding for: '{test_query}' ===")
    print(understanding)

    feats = extractor.extract(understanding, catalog)
    ranked = ltr.rank(feats, catalog, top_k=10)
    print("\n=== Top-10 Ranked Results ===")
    print(ranked[["doc_id", "category", "title", "popularity", "ltr_score"]].to_string(index=False))

    # --- Evaluate NDCG@10 across held-out queries ---
    ndcg_scores = []
    for _, q_row in query_log.sample(30, random_state=1).iterrows():
        understanding = qu.understand(q_row.query_text)
        feats = extractor.extract(understanding, catalog)
        true_relevance = label_relevance(feats)
        predicted_scores = ltr.model.predict(feats)
        order = np.argsort(-predicted_scores)
        ndcg_scores.append(ndcg_at_k(true_relevance.values[order], k=10))

    print(f"\nAverage NDCG@10 across 30 held-out queries: {np.mean(ndcg_scores):.3f}")

=== Query Understanding for: 'wireless headphones' ===
{'clean_query': 'wireless headphones', 'tokens': ['wireless', 'headphones'], 'intent_category': 'Electronics', 'intent_confidence': np.float64(0.6556708976684447)}

=== Top-10 Ranked Results ===
 doc_id    category                                                                         title  popularity  ltr_score
    119 Electronics         laptop speaker usb headphones - durable high quality electronics item    0.444344   3.007564
    279 Electronics         usb speaker bluetooth charger - durable high quality electronics item    0.186442   3.004847
    232 Electronics          wireless watch charger smart - durable high quality electronics item    0.319247   3.002037
    101 Electronics wireless earbuds headphones bluetooth - durable high quality electronics item    0.233861   3.001505
    242 Electronics         earbuds laptop headphones usb - durable high quality electronics item    0.243353   3.000880
    181 Electronics     